# Project 02 · Personal Finance Helper Agent (Reference solution)

**Module 02 · Tools & Your First Agents** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 4–6 hours |
| 📋 Prerequisites | Lessons 02.01–02.07 |
| 🔑 Keys | `GROQ_API_KEY` (live mode) · `LANGSMITH_API_KEY` (tracing, optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |
| 📄 Spec | [project/README.md](../README.md) (also in [COURSE_OUTLINE.md](../../../../COURSE_OUTLINE.md#project-02--personal-finance-helper-agent)) |

**Goal:** an agent that answers questions about your (sample) expense log with safe, deterministic tools, returns a validated `FinanceAnswer`, and never does arithmetic itself.

This is the **reference solution**. Try the [starter notebook](../02-project-personal-finance-agent.ipynb) first; compare afterwards (same cells, same order).

## Requirements (from the spec)

1. Tools: `load_expenses(month)`, `sum_by_category(month)`, `top_merchants(n, month)`, `convert_currency(amount, from_currency, to_currency)` (static rates file). Each has a docstring, typed args and error handling. (We add `mastery.tools.calculator` for differences and ratios.)
2. `create_agent` with `openai/gpt-oss-20b`, a clear system prompt, and `response_format=ToolStrategy(FinanceAnswer)`.
3. Stream tool calls to the notebook. Trace to the LangSmith project `m02-finance-agent`.
4. 10 test questions with expected answers ([README](../README.md#the-10-test-questions)). At least 8 correct in every run (use `N_RUNS = 3` for your final score).

**Acceptance criteria:** no arithmetic by the LLM (every figure, and every number in the answer text, was produced by a tool from backed inputs) · the agent refuses questions outside the data · Restart & Run All passes.

**Modes:** with `GROQ_API_KEY` the notebook is **LIVE**. Without it, it runs in **OFFLINE DEMO** mode: each question is answered by a scripted stand-in model that calls *your* tools in a fixed plan and builds its answer from the answer key, with one deliberate mistake, so you can build and test the tools, checks and scoring for free. Demo scores measure your **tools and checks**, not a model.

## Step 0 · Setup

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2" "pandas>=3.0.6,<4" "matplotlib>=3.11.2,<4" "langsmith>=0.14.7,<1"

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/02-tools-and-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
# Imports and settings used by the whole notebook.
import ast
import json
import re
from functools import lru_cache

import pandas as pd
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ModelRetryMiddleware
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import AIMessage, ToolMessage
from langchain.tools import tool
from langchain_core.tools import ToolException
from typing import Literal

from mastery.agents import agent_stats, format_stream_part, recursion_limit_for
from mastery.config import is_set, load_env, load_settings
from mastery.models import is_tool_use_failed
from mastery.testing import ScriptedChatModel
from mastery.tools import calculator
from mastery.tools.dates import parse_month

load_env()
settings = load_settings(load_dotenv_file=False)
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
MODE = "LIVE" if HAVE_GROQ_KEY else "OFFLINE DEMO"
TRACE_PROJECT = "m02-finance-agent"

DATA_DIR = REPO_ROOT / "data" / "samples" / "expenses"
EXPENSES_CSV = DATA_DIR / "expenses-2026.csv"
RATES_JSON = DATA_DIR / "fx-rates.json"

print("mode            :", MODE)
print("model           :", settings.model_fast)
print("expenses file   :", EXPENSES_CSV.relative_to(REPO_ROOT), "exists" if EXPENSES_CSV.exists() else "MISSING")
print("LangSmith key   :", "set" if HAVE_LANGSMITH_KEY else "missing (tracing off; fine)")

## Step 1 · The data

`data/samples/expenses/expenses-2026.csv`: 128 synthetic expenses, July–September 2026, in COP. `fx-rates.json`: **illustrative static** rates (1 USD = 4,000 COP, 1 EUR = 4,350 COP, 1 MXN = 215 COP). Load once; tools read from memory.

In [ ]:
# Load the data ONCE (cached). Tools read from these, never from paths the model chooses.
@lru_cache(maxsize=1)
def expenses() -> pd.DataFrame:
    df = pd.read_csv(EXPENSES_CSV, encoding="utf-8-sig", dtype={"amount": "int64"})
    df["month"] = df["date"].str[:7]
    return df


@lru_cache(maxsize=1)
def fx_rates() -> dict:
    return json.loads(RATES_JSON.read_text(encoding="utf-8"))


AVAILABLE_MONTHS = sorted(expenses()["month"].unique())
print(len(expenses()), "rows | months:", AVAILABLE_MONTHS, "| currencies:", sorted(fx_rates()["rates_to_base"]))
print(expenses().head(3).to_string(index=False))

In [ ]:
# Shared validation: a clear, model-readable error for months we don't have.
def month_rows(month: str) -> pd.DataFrame:
    """Return the rows for one YYYY-MM month, or raise a ToolException the model can act on."""
    year, mon = parse_month(month)   # Already raises a model-readable ToolException for bad formats.
    key = f"{year:04d}-{mon:02d}"
    if key not in AVAILABLE_MONTHS:
        raise ToolException(f"No data for {key}. Available months: {', '.join(AVAILABLE_MONTHS)}.")
    return expenses()[expenses()["month"] == key]

## Step 2 · The tools

Four tools from the spec. Rules from Lessons 02.01 and 02.04: typed arguments, Google-style docstrings (`parse_docstring=True`), **plain integers in JSON** (easy for the model to copy exactly), `ToolException` with a fix-it message for bad input, and nothing the model can use to read other files.

In [ ]:
@tool(parse_docstring=True, response_format="content_and_artifact")
def load_expenses(month: str) -> tuple[str, list[dict]]:
    """List the user's expenses for one month: count, total and the 10 largest rows.

    Args:
        month: Month in YYYY-MM format, e.g. 2026-09.
    """
    rows = month_rows(month).sort_values("amount", ascending=False)
    top = rows.head(10)[["date", "merchant", "category", "amount"]].to_dict("records")
    content = json.dumps({"month": month, "currency": "COP", "count": int(len(rows)),
                          "total": int(rows["amount"].sum()), "largest_10": top}, ensure_ascii=False)
    return content, rows.drop(columns="month").to_dict("records")


@tool(parse_docstring=True)
def sum_by_category(month: str) -> str:
    """Total spending per category for one month, in COP, largest first, plus the month total.

    Args:
        month: Month in YYYY-MM format, e.g. 2026-09.
    """
    rows = month_rows(month)
    by_cat = rows.groupby("category")["amount"].sum().sort_values(ascending=False)
    return json.dumps({"month": month, "currency": "COP", "by_category": {k: int(v) for k, v in by_cat.items()},
                       "total": int(rows["amount"].sum())})


@tool(parse_docstring=True)
def top_merchants(n: int, month: str) -> str:
    """The n merchants where the user spent the most in one month (summed per merchant), in COP.

    Args:
        n: How many merchants to return (1-10).
        month: Month in YYYY-MM format, e.g. 2026-08.
    """
    if not 1 <= n <= 10:
        raise ToolException("n must be between 1 and 10.")
    rows = month_rows(month)
    top = rows.groupby("merchant")["amount"].sum().sort_values(ascending=False).head(n)
    return json.dumps({"month": month, "currency": "COP",
                       "top_merchants": [{"merchant": m, "total": int(v)} for m, v in top.items()]}, ensure_ascii=False)


Currency = Literal["COP", "USD", "EUR", "MXN"]   # The schema lists the choices; bad codes never reach the code.


@tool(parse_docstring=True)
def convert_currency(amount: float, from_currency: Currency, to_currency: Currency) -> str:
    """Convert an amount between currencies with the course's STATIC rates file (not live rates).

    Args:
        amount: The amount in from_currency, e.g. 5077200.
        from_currency: ISO code of the amount.
        to_currency: ISO code to convert to (must differ from from_currency).
    """
    if from_currency == to_currency:
        raise ToolException("from_currency and to_currency are the same: no conversion needed.")
    rates = fx_rates()["rates_to_base"]
    result = round(amount * rates[from_currency] / rates[to_currency], 2)
    # Only the result: echoing `amount` back would let any number "come from a tool".
    return json.dumps({"from": from_currency, "to": to_currency, "result": result, "rates_as_of": fx_rates()["as_of"]})


for t in (load_expenses, sum_by_category, top_merchants, convert_currency):
    t.handle_tool_error = True   # ToolException → error ToolMessage the model can read (Lesson 02.04).

In [ ]:
# 🧪 Offline: test each tool directly against the answer key BEFORE any model sees it.
TOOLS = [load_expenses, sum_by_category, top_merchants, convert_currency, calculator]


def check(label: str, func, expected) -> None:
    try:
        got = func()
    except Exception as exc:  # Unfinished TODOs land here.
        print(f"❌ {label:<38} {type(exc).__name__}: {exc}")
        return
    print(f"{'✅' if got == expected else '❌'} {label:<38} got {got!r}" + ("" if got == expected else f", expected {expected!r}"))


check("sum_by_category total 2026-09", lambda: json.loads(sum_by_category.invoke({"month": "2026-09"}))["total"], 5_077_200)
check("sum_by_category groceries 2026-09", lambda: json.loads(sum_by_category.invoke({"month": "2026-09"}))["by_category"]["groceries"], 1_694_000)
check("top_merchants #2 2026-08", lambda: json.loads(top_merchants.invoke({"n": 3, "month": "2026-08"}))["top_merchants"][1], {"merchant": "Éxito Calle 80", "total": 1_602_000})
check("load_expenses count 2026-07", lambda: json.loads(load_expenses.invoke({"month": "2026-07"}))["count"], 42)
check("convert 5,077,200 COP → USD", lambda: json.loads(convert_currency.invoke({"amount": 5077200, "from_currency": "COP", "to_currency": "USD"}))["result"], 1269.3)
check("month outside the data → error text", lambda: sum_by_category.invoke({"month": "2026-10"}).split(".")[0], "No data for 2026-10")
check("bad n → error text", lambda: top_merchants.invoke({"n": 50, "month": "2026-08"}), "n must be between 1 and 10.")
check("convert COP → COP → error text", lambda: convert_currency.invoke({"amount": 870400, "from_currency": "COP", "to_currency": "COP"}).split(":")[0], "from_currency and to_currency are the same")
check("convert does not echo the amount", lambda: "870400" in convert_currency.invoke({"amount": 870400, "from_currency": "COP", "to_currency": "EUR"}), False)

**You should now see** nine ✅ lines (in the starter: ❌ with `TODO` until you implement the tools). Test tools without a model first: if a tool is wrong, no prompt will fix the answer.

## Step 3 · The answer schema and the system prompt

`FinanceAnswer` follows the spec, plus `out_of_scope` so refusals are data, not wording (Lesson 02.05). The `Figure.value` description is an instruction: *copied from a tool result*.

In [ ]:
class Figure(BaseModel):
    """One number used in the answer."""

    label: str = Field(description="What the number is, e.g. 'groceries 2026-09'.")
    value: float = Field(description="The number EXACTLY as it appears in a tool result. Never computed by you.")
    unit: str = Field(description="Currency code (COP, USD, EUR, MXN) or 'count'.")


class FinanceAnswer(BaseModel):
    """Final answer to the user's question about their expense log."""

    answer: str = Field(description="One to three sentences for the user, using the figures.")
    figures: list[Figure] = Field(default_factory=list, description="Every number the answer uses, copied from tool results.")
    caveats: list[str] = Field(default_factory=list, description="Limits: static FX rates, missing months, assumptions.")
    out_of_scope: bool = Field(default=False, description="True if the question is not about the user's expense data.")

In [ ]:
SYSTEM_PROMPT = f"""You are a personal-finance assistant for ONE user's expense log.
Data: expenses from {AVAILABLE_MONTHS[0]} to {AVAILABLE_MONTHS[-1]}, amounts in COP. Static FX rates as of {fx_rates()["as_of"]}.

Rules:
1. Get every number from a tool. Never do arithmetic yourself: use the calculator for differences, sums and ratios,
   and convert_currency for currencies. Copy numbers into `figures` exactly as the tools returned them.
2. Pick the most specific tool: sum_by_category for totals and categories, top_merchants for "where/who the most",
   load_expenses for counts and individual expenses.
3. If a month has no data, say so, list the available months, and give no figures.
4. If the question is not about this expense data (investing advice, weather, other people), set out_of_scope=true,
   explain briefly, and give no figures.
5. Mention in caveats when you used static exchange rates."""

In [ ]:
def build_agent(model):
    """The Project 02 agent. ToolStrategy is explicit: Groq can't combine JSON-schema mode with tools (Lesson 02.05)."""
    return create_agent(
        model,
        tools=TOOLS,
        system_prompt=SYSTEM_PROMPT,
        response_format=ToolStrategy(FinanceAnswer),
        middleware=[ModelCallLimitMiddleware(run_limit=10, exit_behavior="end"),
                    # A malformed tool call (Groq "400 tool_use_failed") is a model slip, not a bad request: retry it once.
                    ModelRetryMiddleware(max_retries=1, retry_on=is_tool_use_failed, on_failure="error")],
        name="finance_agent",
    )


# Above the middleware's 10-call budget, or GraphRecursionError fires before the graceful stop (Lesson 02.03).
RUN_CONFIG = {"recursion_limit": recursion_limit_for(build_agent(ScriptedChatModel(responses=["ok"])), 10),
              "tags": ["m02", "project-02"]}
print("recursion_limit:", RUN_CONFIG["recursion_limit"])
print("tools:", [t.name for t in TOOLS])
print("system prompt:", len(SYSTEM_PROMPT), "characters")

## Step 4 · Test questions, checks and grading

In [ ]:
# The 10 test questions and their expected answers (same table as the README).
# kind: "numbers" → every expected value must be among the answer's figures (±0.5 COP, ±0.01 USD/EUR, exact counts);
#       "no_data" → no figures and not out_of_scope; "refuse" → out_of_scope=True and no figures.
QUESTIONS = [
    {"id": 1, "q": "How much did I spend in total in September 2026?", "kind": "numbers", "expected": [5077200], "unit": "COP"},
    {"id": 2, "q": "How much did I spend on groceries in September 2026?", "kind": "numbers", "expected": [1694000], "unit": "COP"},
    {"id": 3, "q": "Which 3 merchants did I spend the most at in August 2026, and how much at each?", "kind": "numbers",
     "expected": [1850000, 1602000, 486200], "unit": "COP"},
    {"id": 4, "q": "Apart from housing, what was my biggest spending category in July 2026?", "kind": "numbers", "expected": [1848600], "unit": "COP"},
    {"id": 5, "q": "How many expenses did I record in July 2026?", "kind": "numbers", "expected": [42], "unit": "count"},
    {"id": 6, "q": "What was my total September 2026 spending in US dollars?", "kind": "numbers", "expected": [1269.3], "unit": "USD"},
    {"id": 7, "q": "How much more did I spend in August 2026 than in July 2026?", "kind": "numbers", "expected": [870400], "unit": "COP"},
    {"id": 8, "q": "How much did I spend on restaurants in September 2026, in euros?", "kind": "numbers", "expected": [96.0], "unit": "EUR"},
    {"id": 9, "q": "How much did I spend in October 2026?", "kind": "no_data", "expected": []},
    {"id": 10, "q": "Should I put my savings into bitcoin?", "kind": "refuse", "expected": []},
]
pd.DataFrame(QUESTIONS)[["id", "q", "kind", "expected", "unit"]]

**"No arithmetic by the LLM"** is checkable, if the check is strict about *where* a number came from. `backed_numbers()` walks the run in order: data tools back their numeric JSON values; `convert_currency` backs its result only if its `amount` was already backed; the calculator backs its result only if it did real arithmetic on backed operands. So the model can't launder a number by typing `calculator("870400")`. `unbacked_prose()` applies the same rule to the answer text (years and numbers from the question are allowed).

In [ ]:
NUMBER = re.compile(r"-?\d+(?:\.\d+)?")
TOLERANCE = {"COP": 0.5, "USD": 0.01, "EUR": 0.01, "MXN": 0.01, "count": 0}   # From the QUESTION's unit, not the model's.


def close(a: float, b: float, tol: float = 0.01) -> bool:
    return abs(a - b) <= tol


def json_numbers(value) -> list[float]:
    """Numeric VALUES in a parsed JSON tool result (not digits inside strings such as '2026-09')."""
    if isinstance(value, bool):
        return []
    if isinstance(value, (int, float)):
        return [float(value)]
    if isinstance(value, dict):
        return [x for v in value.values() for x in json_numbers(v)]
    if isinstance(value, list):
        return [x for v in value for x in json_numbers(v)]
    return []


def calc_operands(expression: str) -> list[float] | None:
    """Numbers typed into a calculator expression, or None if it does no arithmetic (a bare echo like '870400')."""
    try:
        tree = ast.parse(expression.replace("^", "**"), mode="eval")
    except SyntaxError:
        return None
    if not any(isinstance(n, ast.BinOp) for n in ast.walk(tree)):
        return None
    return [float(n.value) for n in ast.walk(tree) if isinstance(n, ast.Constant) and isinstance(n.value, (int, float))]


def backed_numbers(messages) -> list[float]:
    """Numbers a tool really produced, walking the run in order.

    - data tools (load_expenses, sum_by_category, top_merchants): every numeric JSON value;
    - convert_currency: its result, only if `amount` was already backed;
    - calculator: its result, only if it did arithmetic and every operand was backed
      (small constants up to 100, like `* 100` for a percentage, are allowed).
    """
    calls = {tc["id"]: tc for m in messages if isinstance(m, AIMessage) for tc in m.tool_calls}
    backed: list[float] = []
    for m in messages:
        if not isinstance(m, ToolMessage) or m.status == "error" or m.name == "FinanceAnswer":
            continue
        args = calls.get(m.tool_call_id, {}).get("args", {})
        is_backed = lambda x: abs(x) <= 100 or any(close(x, b) for b in backed)
        if m.name == "calculator":
            operands = calc_operands(str(args.get("expression", "")))
            if operands is not None and all(is_backed(x) for x in operands):
                backed += [float(m.content)]
            continue
        try:
            data = json.loads(m.content)
        except (TypeError, ValueError):
            continue
        if m.name == "convert_currency":
            if any(close(float(args.get("amount", "nan")), b) for b in backed):
                backed += [float(data["result"])]
            continue
        backed += json_numbers(data)
    return backed


# Thousands separators: "1,850,000", "1 850 000" with a (narrow) no-break space, as models often write COP.
GROUPS = re.compile(r"(?<=\d)[,\u00a0\u202f\u2009 ](?=\d{3}(?!\d))")


# Dates are not amounts: "2026-07", "2026‑10‑01" (any dash) would otherwise read as 2026 and 7.
DATES = re.compile(r"\b\d{4}[-\u2010\u2011\u2012\u2013\u2014]\d{1,2}(?:[-\u2010\u2011\u2012\u2013\u2014]\d{1,2})?\b")


def prose_numbers(answer_text: str) -> list[float]:
    return [float(x) for x in NUMBER.findall(GROUPS.sub("", DATES.sub(" ", answer_text)))]


def invented_numbers(result) -> list[float]:
    """Figures no tool produced: evidence the model did arithmetic, guessed, or laundered a number through a tool."""
    answer = result.get("structured_response")
    if answer is None:
        return []
    seen = backed_numbers(result["messages"])
    return [f.value for f in answer.figures if not any(close(f.value, s) for s in seen)]


def unbacked_prose(result, question: str) -> list[float]:
    """Numbers in the answer TEXT that are neither a figure, a tool number, a year, nor in the question."""
    answer = result.get("structured_response")
    if answer is None:
        return []
    # Digits inside tool TEXT count too: a merchant called "Éxito Calle 80" is data, not a made-up number.
    tool_text = [x for m in result["messages"]
                 if isinstance(m, ToolMessage) and m.name != "FinanceAnswer" for x in prose_numbers(m.text)]   # not the echo
    ok = backed_numbers(result["messages"]) + tool_text + [f.value for f in answer.figures] + prose_numbers(question)
    return [x for x in prose_numbers(answer.answer) if not (1900 <= x <= 2100 or any(close(x, o) for o in ok))]

In [ ]:
def grade(spec: dict, result) -> tuple[bool, str]:
    """Is the structured answer correct for this question? Returns (ok, reason)."""
    answer = result.get("structured_response")
    if answer is None:
        return False, "no structured_response (limit hit or error)"
    values = [f.value for f in answer.figures]
    if spec["kind"] == "refuse":
        return (answer.out_of_scope and not values), f"out_of_scope={answer.out_of_scope}, figures={values}"
    if spec["kind"] == "no_data":
        return (not answer.out_of_scope and not values), f"figures={values}"
    tol = TOLERANCE[spec["unit"]]
    missing = [e for e in spec["expected"] if not any(close(e, v, tol) for v in values)]
    if missing:
        return False, f"missing {missing}, got {values}"
    seen = backed_numbers(result["messages"])
    unbacked = [e for e in spec["expected"] if not any(close(e, s, tol) for s in seen)]   # Right number, but no tool produced it?
    return (not unbacked), ("all expected figures present" if not unbacked else f"{unbacked} not in any tool result")


# 🧪 Offline: the grader itself, on hand-made answers.
fake = {"structured_response": FinanceAnswer(answer="x", figures=[Figure(label="t", value=5077200, unit="COP")]),
        "messages": [ToolMessage('{"total": 5077200}', tool_call_id="t1", name="sum_by_category")]}
print(grade(QUESTIONS[0], fake))
print(grade(QUESTIONS[9], {"structured_response": FinanceAnswer(answer="Not about your data.", out_of_scope=True), "messages": []}))

**You should now see** `(True, 'all expected figures present')` and `(True, 'out_of_scope=True, figures=[]')`.

## Step 5 · Run the agent

In [ ]:
# OFFLINE DEMO stand-ins: a fixed tool plan per question, and a final FinanceAnswer from the answer key.
# Question 7 is answered WITHOUT the calculator and with a wrong number (deliberate), so your
# invented_numbers() check has something to catch.
PLANS = {
    1: [("sum_by_category", {"month": "2026-09"})],
    2: [("sum_by_category", {"month": "2026-09"})],
    3: [("top_merchants", {"n": 3, "month": "2026-08"})],
    4: [("sum_by_category", {"month": "2026-07"})],
    5: [("load_expenses", {"month": "2026-07"})],
    6: [("sum_by_category", {"month": "2026-09"}), ("convert_currency", {"amount": 5077200, "from_currency": "COP", "to_currency": "USD"})],
    7: [("sum_by_category", {"month": "2026-08"}), ("sum_by_category", {"month": "2026-07"})],
    8: [("sum_by_category", {"month": "2026-09"}), ("convert_currency", {"amount": 417600, "from_currency": "COP", "to_currency": "EUR"})],
    9: [("sum_by_category", {"month": "2026-10"})],
    10: [],
}


def demo_model(spec: dict) -> ScriptedChatModel:
    calls = [AIMessage("", tool_calls=[{"name": n, "args": a, "id": f"q{spec['id']}c{i}"}]) for i, (n, a) in enumerate(PLANS[spec["id"]])]
    values = [870000] if spec["id"] == 7 else spec["expected"]          # Deliberate mistake on question 7.
    unit = "USD" if spec["id"] == 6 else "EUR" if spec["id"] == 8 else "count" if spec["id"] == 5 else "COP"
    final = {
        "answer": f"[demo] {spec['q']}",
        "figures": [{"label": f"value {i + 1}", "value": v, "unit": unit} for i, v in enumerate(values)],
        "caveats": ["static FX rates"] if unit in ("USD", "EUR") else [],
        "out_of_scope": spec["kind"] == "refuse",
    }
    return ScriptedChatModel(responses=[*calls, AIMessage("", tool_calls=[{"name": "FinanceAnswer", "args": final, "id": f"q{spec['id']}f"}])])


def model_for(spec: dict):
    if HAVE_GROQ_KEY:
        from mastery.models import get_model
        return get_model("fast", temperature=0, reasoning_effort="low")
    return demo_model(spec)


print("mode:", MODE)

In [ ]:
# 🧪 Offline: three ways to "pass" question 7 without doing the work. A good check flags all three.
gaming = {
    "calculator echo": [("sum_by_category", {"month": "2026-08"}), ("calculator", {"expression": "870400"})],
    "COP → COP conversion": [("convert_currency", {"amount": 870400, "from_currency": "COP", "to_currency": "COP"})],
    "made-up numbers in the prose": [("sum_by_category", {"month": "2026-08"}), ("sum_by_category", {"month": "2026-07"}),
                                     ("calculator", {"expression": "5837600 - 4967200"})],
}
q7 = QUESTIONS[6]
gaming_caught = 0
for name, plan in gaming.items():
    calls = [AIMessage("", tool_calls=[{"name": n, "args": a, "id": f"g{i}"}]) for i, (n, a) in enumerate(plan)]
    text = "You spent 870,400 COP more, about 18% more, so 3 million less is left." if "prose" in name else "870,400 COP more."
    final = {"answer": text, "figures": [{"label": "difference", "value": 870400, "unit": "COP"}], "caveats": [], "out_of_scope": False}
    stand_in = ScriptedChatModel(responses=[*calls, AIMessage("", tool_calls=[{"name": "FinanceAnswer", "args": final, "id": "gf"}])])
    result = build_agent(stand_in).invoke({"messages": [{"role": "user", "content": q7["q"]}]}, RUN_CONFIG)
    caught = invented_numbers(result) or unbacked_prose(result, q7["q"])
    gaming_caught += bool(caught)
    print(("✅ flagged    " if caught else "❌ NOT flagged") + f" {name:<30} invented={invented_numbers(result)} prose={unbacked_prose(result, q7['q'])}")

**You should now see** three ✅ `flagged` lines in the solution: a calculator that only echoes `870400`, a currency "conversion" from COP to COP, and a correct figure wrapped in prose with made-up numbers. In the starter, the first two stay ❌ `NOT flagged` until TODO 6 is done (the prose check is given, so the third is already caught). A check you can pass without doing the work is not a check.

In [ ]:
# 💸 in LIVE mode (🧪 in DEMO mode): stream ONE question and show each tool call as it happens.
import langsmith as ls

spec = QUESTIONS[6]   # "How much more did I spend in August than in July?" needs several tools.
agent = build_agent(model_for(spec))
final_state = None

with ls.tracing_context(enabled=HAVE_LANGSMITH_KEY, project_name=TRACE_PROJECT):
    for part in agent.stream({"messages": [{"role": "user", "content": spec["q"]}]}, RUN_CONFIG,
                             stream_mode=["updates", "values"], version="v2"):
        if part["type"] == "values":
            final_state = part["data"]
        elif (line := format_stream_part(part, max_chars=140)):
            print(line)

print("\nstructured_response:", final_state.get("structured_response"))
print("stats:", agent_stats(final_state["messages"]))

**You should now see** each step as it happens: `[model] 🤖 → sum_by_category(month='2026-08')`, the tool result, the July lookup, (LIVE: a `calculator` call), and the final `FinanceAnswer`. In DEMO mode the answer is `870000`, the deliberate mistake: the true difference is **870,400**, and no tool ever returned 870000.

With `LANGSMITH_API_KEY` set, the run is in LangSmith under the project **`m02-finance-agent`**.

In [ ]:
# 💸 in LIVE mode (~25–40 short calls per run, free tier): run all 10 questions N_RUNS times and grade them.
# One run is a sample, not a score: use N_RUNS = 3 for your final number (~75–120 calls).
N_RUNS = 1


def run_once(spec: dict, run: int) -> dict:
    try:
        result = build_agent(model_for(spec)).invoke({"messages": [{"role": "user", "content": spec["q"]}]},
                                                     {**RUN_CONFIG, "run_name": f"q{spec['id']}-run{run}"})
    except Exception as exc:  # One failure must not stop the evaluation.
        return {"id": spec["id"], "run": run, "correct": False, "invented": "-", "prose": "-", "tools": "-",
                "detail": f"{type(exc).__name__}: {exc}"[:120]}
    ok, detail = grade(spec, result)
    invented, prose = invented_numbers(result), unbacked_prose(result, spec["q"])
    stats = agent_stats(result["messages"])
    return {"id": spec["id"], "run": run, "correct": ok and not invented and not prose, "invented": invented or "",
            "prose": prose or "", "tools": ", ".join(t for t in stats["tools_used"] if t != "FinanceAnswer"), "detail": detail}


with ls.tracing_context(enabled=HAVE_LANGSMITH_KEY, project_name=TRACE_PROJECT):
    runs = pd.DataFrame([run_once(spec, r) for r in range(1, N_RUNS + 1) for spec in QUESTIONS])

per_run = runs.groupby("run")["correct"].sum()
n_correct = int(per_run.min())           # Acceptance uses the WORST run, not the best.
scores = runs.groupby("id").agg(pass_rate=("correct", "mean"), invented=("invented", "last"),
                                prose=("prose", "last"), tools=("tools", "last"), detail=("detail", "last"))
print(f"{MODE}: {N_RUNS} run(s) · correct per run {per_run.tolist()} · mean {per_run.mean():.1f} · min {n_correct}/10 (target ≥ 8)")
scores

In [ ]:
# Acceptance checks (spec): ≥ 8 correct in the worst run, no invented numbers, refusal works.
checks = {
    "≥ 8 of 10 correct (worst run)": n_correct >= 8,
    "no invented numbers": (runs["invented"] == "").all(),
    "no unbacked numbers in the answer text": (runs["prose"] == "").all(),
    "refuses out-of-scope (Q10, every run)": bool(scores.loc[10, "pass_rate"] == 1),
    "all 3 gaming attempts flagged": gaming_caught == len(gaming),
}
for name, passed in checks.items():
    print(("✅" if passed else "❌"), name)
if MODE == "OFFLINE DEMO":
    print("\n(OFFLINE DEMO: Q7 is a deliberate mistake. Once your tools and checks work: 9/10, and 'no invented numbers' is ❌ because of Q7.)")

**You should now see** the score table (one row per question, `pass_rate` over `N_RUNS` runs) and the acceptance checks. **LIVE:** aim for ≥ 8/10 with no invented numbers. If a question fails, open its trace (`run_name` = `q<id>`) and ask: wrong tool? wrong arguments? number re-typed instead of copied? Then fix the **tool descriptions or the prompt**, not the grader.

**DEMO** (with your tools and TODO 6 done): 9/10, Q7 flagged with `invented: [870000.0]`.

## Step 6 · Stretch goals

1. **Chart tool:** save a PNG per month (below, offline). Then add it to `TOOLS` and ask *"Make a chart of my September spending"*.
2. **Dynamic model:** route comparison questions to `openai/gpt-oss-120b` with `@wrap_model_call` (Lesson 02.03). Check the price per million tokens in `config/models.yaml` and compare the score.

In [ ]:
# Stretch 1 (🧪 offline): a chart tool that saves a PNG and returns its path, not the image.
import matplotlib

matplotlib.use("Agg")                     # No window: works in notebooks, scripts and servers.
import matplotlib.pyplot as plt

CHART_DIR = REPO_ROOT / "data" / "outputs" / "m02"   # data/outputs/ is git-ignored.


@tool(parse_docstring=True)
def chart_by_category(month: str) -> str:
    """Save a bar chart of spending per category for one month as a PNG and return its file path.

    Args:
        month: Month in YYYY-MM format, e.g. 2026-09.
    """
    by_cat = month_rows(month).groupby("category")["amount"].sum().sort_values()
    CHART_DIR.mkdir(parents=True, exist_ok=True)
    path = CHART_DIR / f"spending-{month}.png"
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(by_cat.index, by_cat.values / 1000)
    ax.set_xlabel("thousand COP")
    ax.set_title(f"Spending by category, {month}")
    fig.tight_layout()
    fig.savefig(path, dpi=100)
    plt.close(fig)
    return f"Saved chart: {path.relative_to(REPO_ROOT).as_posix()}"


chart_by_category.handle_tool_error = True
print(chart_by_category.invoke({"month": "2026-09"}))

In [ ]:
# Stretch 2 (💸): use gpt-oss-120b for "complex" questions (comparisons, several months), 20b otherwise.
from langchain.agents.middleware import ModelRequest, wrap_model_call

COMPLEX = re.compile(r"\b(more|less|than|compare|difference|change|versus|vs)\b", re.I)

if HAVE_GROQ_KEY:
    from mastery.models import get_model

    big = get_model("reasoning", temperature=0, reasoning_effort="low")

    @wrap_model_call
    def complex_to_120b(request: ModelRequest, handler):
        question = next((m.text for m in request.messages if m.type == "human"), "")
        return handler(request.override(model=big) if COMPLEX.search(question) else request)

    dyn_agent = create_agent(get_model("fast", temperature=0, reasoning_effort="low"), tools=[*TOOLS, chart_by_category],
                             system_prompt=SYSTEM_PROMPT, response_format=ToolStrategy(FinanceAnswer),
                             middleware=[complex_to_120b, ModelCallLimitMiddleware(run_limit=10)])
    out = dyn_agent.invoke({"messages": [{"role": "user", "content": QUESTIONS[6]["q"]}]}, RUN_CONFIG)
    print({m.response_metadata.get("model_name") for m in out["messages"] if isinstance(m, AIMessage)})
    print(out["structured_response"])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set. Rule check only:",
          [bool(COMPLEX.search(s["q"])) for s in QUESTIONS])

## Findings (write yours)

- Score (LIVE): __/10. Which questions failed, and why (tool choice, arguments, copying)?
- Did the model ever do arithmetic itself? What in the prompt or schema stopped it?
- Did `out_of_scope` work for Q10? Try three more out-of-scope questions.
- One change you'd make before giving this agent real bank data (Lesson 02.04's security rules).

Log the decision (model, prompt version, score) in [`docs/decision-log.md`](../../../../docs/decision-log.md).

## ✅ Before you commit

- [ ] Restart & Run All passes (LIVE and without a key).
- [ ] All TODOs done; ≥ 8/10 LIVE in the worst of 3 runs (`N_RUNS = 3`); no invented numbers; gaming attempts flagged.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); no keys in outputs.

---
## ➡️ What's next

**Next:** [Module 03 · Memory & Context Engineering](../../../03-memory-and-context/README.md). Start with its **Before you start this module** block (pull, install, verify).